# Input Grad Steering — transformer · window 16

**Question.** Can the gradient of a frozen linear position probe, backpropagated **through the transformer to the
input observation**, produce a *semantically* edited observation (the object's bump moves toward the teleport
target) — or does it find an adversarial perturbation that flips the readout without producing anything the
dynamics respect? For the transformer this matters doubly: the observation buffer is its only **persistent** state
channel (`../METRICS_AND_EDITORS.md`, architecture-specific editors), so this is a training-free, probe-only write
to the carried state.

**Data / model provenance.** Model `W16` = **transformer · window 16** (d_model 256, 4 layers, RoPE, `state_span`
61 → at `ef`=20 the effective carried state is all 20 frames), checkpoint `runs/transformers/W16/best_model.pt`,
row copied from `../transformers/TRANSFORMER_RUNS.md`. Dataset `datasets/4_fixed_refl_inview` (2 objects, T=40,
R=128, obs noise 0.2, edit frame `ef`=20, in-frustum teleports). N=64 edits, K=15 rollout steps. §4 metrics
imported from `scripts/editability_metrics.py`; probes from `pim.extractors.fit_readability_probes`.

## Definitions

| term / metric | definition | units | better |
|---|---|---|---|
| **residual point ℓ** | the residual stream read by probes/steering: 0 = encoder port `relu(Linear(obs))`, 1–3 = input to block 2–4, 4 = final pre-LayerNorm stream the decoder reads (labels per `../transformers/TRANSFORMER_RUNS.md`) | — | — |
| **linear / MLP probe R²** | held-out R² of the standard readability probes (`fit_readability_probes`: linear lstsq + 2×256 ReLU MLP, same 80/20 by-**sequence** split), target = positions of both objects (4 dims) | — | ↑ |
| **Input Grad Steering @Lℓ (n, λ)** | Adam (300 steps, lr 0.02) on δ added to the newest **n** history frames, minimizing `‖A_ℓ·state + b_ℓ − target‖² + λ‖δ‖²`, input clamped to [0,1]; `(A_ℓ, b_ℓ)` = the frozen standard **linear** probe at ℓ; target = edited object → teleport target, other object → its true `ef` position (sim units) | — | — |
| **Render write @1 (oracle)** | same write surface, oracle content: newest history frame replaced by the clean render of the edited world (`gt_edited`) | — | — |
| **Δ_true (true edit direction)** | `gt_edited − obs[ef−1]`: clean edited-world render at `ef` minus the actual (noisy) frame being steered. Impurities stated in `README.md`: contains the base frame's noise realisation, and is offset by one frame of motion | intensity | — |
| **cos(δ, Δ_true) / angle** | per-sample cosine (and its angle in degrees) between the steering perturbation and Δ_true; chance = empirical shuffled-pair control `cos(δ_i, Δ_true_j)`, i≠j | — / ° | ↑ / ↓ |
| **probe residual** | `‖A·state + b − target‖` after steering, mean over samples — did the optimization even drive the readout | sim units | ↓ |
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)` on the rays where the two GT worlds differ; +1 = edited world, −1 = unedited, ≈0 = equidistant/garbage (`editability_metrics.py`) | — | ↑ |
| **Target / Ghost / Collateral / Edit-frame RMSE** | RMSE vs the clean edited-world render at rollout step 0, restricted to the target / vacated / other-object / all rays | intensity | ↓ |
| **GT-traj RMSE** | mean RMSE of the K-step free-run vs `clean_obs[ef:ef+K]` | intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)`; >1 = the edit degraded the model | — | ↓ |

All observation-space errors are scored against the **clean** render (`clean_obs`), never the noisy `obs`
(registry rule, 2026-08-04). Rollout step 0 decodes frame `ef` (no shared teacher-forced row anywhere).

In [ ]:
# [1] Setup: model, dataset, constants.
import os, sys
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models.loader import load_checkpoint, load_dataset
from pim.extractors import fit_readability_probes
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ = 2
N_EDIT, K, N_PROBE = 64, 15, 800
STEER_STEPS, STEER_LR = 300, 0.02
LAM_MAIN = 0.1                      # main ridge weight on ||delta||^2
OUT = "/tmp/input_grad_steering_transformer"; os.makedirs(OUT, exist_ok=True)

MODEL_LABEL = "transformer · window 16"
model, info = load_checkpoint("../../../../runs/transformers/W16/best_model.pt", device=DEVICE)
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
test, edits = bundle.test, bundle.edits
ef, R = edits.edit_frame, edits.obs_res
sim = test.config["dataset"]["sim"]
N_RESID = model.cfg.n_layers + 1
RESID_LABELS = ["0 · encoder port", "1 · early", "2 · middle", "3 · late", "4 · last (decoder input)"]
print(f"model {MODEL_LABEL} | epoch {info.epoch} val {info.val_loss:.5f} | d_model {model.cfg.d_model} "
      f"state_span {model.state_span} | ef={ef} R={R} | N_EDIT={N_EDIT} K={K} device={DEVICE}")

In [ ]:
# [2] Standard readability probes at every residual point (the steering probes are the linear ones, frozen).
obs_probe = torch.from_numpy(test.obs[:N_PROBE]).float().to(DEVICE)
Tm1 = test.obs.shape[1] - 1
P_tgt = test.positions[:N_PROBE, :Tm1].reshape(N_PROBE, Tm1, N_OBJ * 2).astype(np.float32)
vis = test.is_visible[:N_PROBE, :Tm1, :N_OBJ].all(axis=2)

model.state_view = "activations"
PROBES = {}
for L in range(N_RESID):
    model.probe_layer = L
    with torch.no_grad():
        _, acts = model.observe_sequence(obs_probe)
    PROBES[L] = fit_readability_probes(acts.cpu().numpy(), P_tgt, mask=vis, device=DEVICE)
model.state_view = "obs_window"

rows = "\n".join(
    f"| {RESID_LABELS[L]} | {PROBES[L]['linear_r2']:.3f} | {PROBES[L]['mlp_r2']:.3f} |"
    for L in range(N_RESID))
display(Markdown("**Held-out position R² per residual point** (standard probes, "
                 f"{PROBES[0]['n_train_seq']}/{PROBES[0]['n_heldout_seq']} train/held-out sequences)\n\n"
                 "| residual point | linear R² | MLP R² |\n|---|---|---|\n" + rows))

# Fig 1 — probe R² by residual point (horizontal bars: long labels)
fig, ax = plt.subplots(figsize=(7, 3.2))
y = np.arange(N_RESID)
ax.barh(y - 0.18, [PROBES[L]["linear_r2"] for L in range(N_RESID)], height=0.36,
        color="#0072B2", label="linear probe")
ax.barh(y + 0.18, [PROBES[L]["mlp_r2"] for L in range(N_RESID)], height=0.36,
        color="#E69F00", label="MLP probe")
ax.set_yticks(y); ax.set_yticklabels(RESID_LABELS); ax.invert_yaxis()
ax.set_xlabel("held-out position R²"); ax.set_xlim(0, 1)
ax.set_title(f"Fig 1 — held-out position R² by residual point ({MODEL_LABEL})")
ax.legend(loc="lower right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_probe_r2.png", dpi=150); plt.show()

In [ ]:
# [3] §4 machinery: edit zones (both GT worlds + ray masks), history, references (unsteered + oracle).
N = min(N_EDIT, edits.n_samples)
oe = edits.edit_object[:N].astype(int)
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef - 1, :N_OBJ, :].astype(np.float32)
pre_pos = edits.positions[:N, ef - 1, :N_OBJ, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)   # edited object already teleported
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ * 2)).float().to(DEVICE)
gt_roll = edits.clean_obs[:N, ef:ef + K, :].astype(np.float32)

ZONES = build_edit_zones(pre_pos=pre_pos, tgt_pos=tgt_pos, pre_vel=pre_vel,
                         edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef + K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)
teleport = ZONES.teleport
gt_edited_t = torch.from_numpy(ZONES.gt_edited).float().to(DEVICE)

# history the model consumed before the edit: frames 0..ef-1 (noisy, as teacher-forced)
hist = torch.from_numpy(edits.obs[:N, :ef]).float().to(DEVICE)     # (N, ef, R)
base_frame = hist[:, -1]                                            # obs[ef-1], the steered frame
delta_true = ZONES.gt_edited - base_frame.cpu().numpy()             # (N, R) the true edit direction

@torch.no_grad()
def rollout_from(state, k=K):
    """Free-run: step 0 decodes frame ef; predictions fed back through the window."""
    x = model.decode(state); out = [x]; s = state
    for _ in range(k - 1):
        x, s = model.step(x, s)
        out.append(x)
    return torch.stack(out, 1).cpu().numpy()

state_unsteered = model.state_from_obs(hist)
CARDS, ROLLS = {}, {}
ROLLS["Unsteered"] = rollout_from(state_unsteered)
CARDS["Unsteered"] = edit_scorecard(ROLLS["Unsteered"], ZONES, gt_roll)

# oracle on the SAME write surface: newest history frame <- clean render of the edited world
hist_oracle = torch.cat([hist[:, :-1], gt_edited_t.unsqueeze(1)], dim=1)
ROLLS["Render write @1 (oracle)"] = rollout_from(model.state_from_obs(hist_oracle))
CARDS["Render write @1 (oracle)"] = edit_scorecard(ROLLS["Render write @1 (oracle)"], ZONES, gt_roll)

print(f"N={N} edits | mean teleport {teleport.mean():.2f} sim-units | "
      f"unsteered Edit Index {CARDS['Unsteered']['edit_index']:+.2f} | "
      f"oracle render-write Edit Index {CARDS['Render write @1 (oracle)']['edit_index']:+.2f}")

In [ ]:
# [4] The editor: Input Grad Steering @Lℓ (n, λ) — Adam on δ over the newest n history frames.
def input_grad_steer(L, n=1, lam=LAM_MAIN, steps=STEER_STEPS, lr=STEER_LR):
    """Returns dict with the steered history, per-sample δ on the newest frame, the raw first
    gradient direction, the steered state, and probe residuals before/after."""
    A = torch.from_numpy(PROBES[L]["A"]).to(DEVICE)   # (4, d_model)
    b = torch.from_numpy(PROBES[L]["b"]).to(DEVICE)
    model.probe_layer = L
    delta = torch.zeros(N, n, R, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    first_grad = None
    for it in range(steps):
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        z = model._activations(model.state_from_obs(frames))          # differentiable
        probe_loss = ((z @ A.T + b - target4) ** 2).sum(-1).mean()
        loss = probe_loss + lam * (delta ** 2).sum((-1, -2)).mean()
        opt.zero_grad(); loss.backward()
        if it == 0:
            first_grad = -delta.grad[:, -1].detach().clone()
        opt.step()
    with torch.no_grad():
        frames = torch.cat([hist[:, :ef - n], (hist[:, ef - n:] + delta).clamp(0, 1)], dim=1)
        state = model.state_from_obs(frames)
        read = model._activations(state) @ A.T + b
        resid_after = (read - target4).norm(dim=-1).mean().item()
        read0 = model._activations(state_unsteered) @ A.T + b
        resid_before = (read0 - target4).norm(dim=-1).mean().item()
    d_eff = (frames[:, -1] - base_frame).cpu().numpy()                # effective δ on the steered frame, post-clamp
    return dict(frames=frames, state=state, delta=d_eff, first_grad=first_grad.cpu().numpy(),
                resid_before=resid_before, resid_after=resid_after)

ARMS = {}
for L in range(N_RESID):
    ARMS[f"Input Grad @L{L} (n=1, λ={LAM_MAIN})"] = input_grad_steer(L, n=1, lam=LAM_MAIN)
for lam in [0.0, 0.01, 1.0]:
    ARMS[f"Input Grad @L4 (n=1, λ={lam})"] = input_grad_steer(4, n=1, lam=lam)
ARMS[f"Input Grad @L4 (n=all {ef}, λ={LAM_MAIN})"] = input_grad_steer(4, n=ef, lam=LAM_MAIN)

for name, a in ARMS.items():
    ROLLS[name] = rollout_from(a["state"])
    CARDS[name] = edit_scorecard(ROLLS[name], ZONES, gt_roll)
    print(f"{name:44s} probe residual {a['resid_before']:.3f} → {a['resid_after']:.3f} sim-units | "
          f"‖δ‖ (newest frame) {np.linalg.norm(a['delta'], axis=-1).mean():.3f} "
          f"(ref ‖Δ_true‖ {np.linalg.norm(delta_true, axis=-1).mean():.3f})")

In [ ]:
# [5] Fig 2 — what the gradient DOES to the observation (the headline question).
#     Rows: 3 large-teleport samples. Columns: residual points 0 / 2 / 4 (all n=1, λ=0.1).
SHOW_L = [0, 2, 4]
SAMPLES = list(np.argsort(teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])
x_axis = np.arange(R)

fig, axes = plt.subplots(len(SAMPLES), len(SHOW_L), figsize=(4.6 * len(SHOW_L), 2.9 * len(SAMPLES)),
                         sharex=True, sharey=True)
for r, smp in enumerate(SAMPLES):
    for c, L in enumerate(SHOW_L):
        ax = axes[r, c]
        arm = ARMS[f"Input Grad @L{L} (n=1, λ={LAM_MAIN})"]
        steered = base_frame[smp].cpu().numpy() + arm["delta"][smp]
        ax.plot(x_axis, base_frame[smp].cpu().numpy(), color="#999999", lw=1.0,
                label="original obs[ef−1] (noisy)")
        ax.plot(x_axis, ZONES.gt_edited[smp], color="#009E73", lw=1.4, ls="--",
                label="clean edited-world render (target look)")
        ax.plot(x_axis, steered, color="#D55E00", lw=1.4, label="steered observation")
        ax.fill_between(x_axis, 0, 1, where=ZONES.target[smp], color="#009E73", alpha=0.08)
        ax.fill_between(x_axis, 0, 1, where=ZONES.ghost[smp], color="#FF5252", alpha=0.08)
        ax.set_ylim(-0.05, 1.1)
        if r == 0:
            ax.set_title(RESID_LABELS[L], fontsize=10)
        if c == 0:
            ax.set_ylabel(f"sample {smp}\n(teleport {teleport[smp]:.1f})\nintensity", fontsize=9)
        if r == len(SAMPLES) - 1:
            ax.set_xlabel("ray")
        style_ax(ax)
handles, labels = axes[0, 0].get_legend_handles_labels()
handles += [Line2D([0], [0], color="#009E73", lw=6, alpha=0.2), Line2D([0], [0], color="#FF5252", lw=6, alpha=0.2)]
labels += ["target rays (shaded)", "ghost rays (shaded)"]
fig.legend(handles, labels, loc="upper center", ncol=5, fontsize=9, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle(f"Fig 2 — steered observation vs the true edited-world render, by residual point "
             f"({MODEL_LABEL}, Input Grad n=1, λ={LAM_MAIN})", y=1.06, fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(f"{OUT}/fig2_steered_obs.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [6] Fig 3 — is the gradient direction the true edit direction? cos(δ, Δ_true) + angle, with shuffled-pair chance.
def cos_rows(vecs):
    """Per-sample cosine of (N, R) vecs against delta_true; returns (mean_cos, mean_angle_deg)."""
    num = (vecs * delta_true).sum(-1)
    den = np.linalg.norm(vecs, axis=-1) * np.linalg.norm(delta_true, axis=-1) + 1e-12
    c = num / den
    return float(c.mean()), float(np.degrees(np.arccos(np.clip(c, -1, 1))).mean())

# empirical chance: mismatched pairs (roll the sample axis)
arm4 = ARMS[f"Input Grad @L4 (n=1, λ={LAM_MAIN})"]
shuf = np.roll(arm4["delta"], 1, axis=0)
cos_chance, _ = cos_rows(shuf)

arm_names = [f"Input Grad @L{L} (n=1, λ={LAM_MAIN})" for L in range(N_RESID)]
stats = []
for name in arm_names:
    c_opt, a_opt = cos_rows(ARMS[name]["delta"])
    c_g, a_g = cos_rows(ARMS[name]["first_grad"])
    stats.append((name, c_opt, a_opt, c_g, a_g))

rows = "\n".join(f"| {RESID_LABELS[i]} | {c:+.3f} | {a:.0f}° | {cg:+.3f} | {ag:.0f}° |"
                 for i, (_, c, a, cg, ag) in enumerate(stats))
display(Markdown("**Alignment of the steering perturbation with the true edit direction** "
                 f"(shuffled-pair chance cosine = {cos_chance:+.3f})\n\n"
                 "| residual point | cos(δ*, Δ_true) | angle | cos(first grad, Δ_true) | angle |\n"
                 "|---|---|---|---|---|\n" + rows))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
y = np.arange(N_RESID)
ax.barh(y - 0.18, [s[1] for s in stats], height=0.36, color="#0072B2", label="converged δ*")
ax.barh(y + 0.18, [s[3] for s in stats], height=0.36, color="#E69F00", label="raw first gradient")
ax.axvline(cos_chance, color="#555555", ls=":", lw=1.4, label="shuffled-pair chance")
ax.set_yticks(y); ax.set_yticklabels(RESID_LABELS); ax.invert_yaxis()
ax.set_xlabel("cosine with the true edit direction Δ_true"); ax.set_xlim(-0.2, 1.0)
ax.set_title(f"Fig 3 — cos(δ, Δ_true) by residual point ({MODEL_LABEL}, n=1, λ={LAM_MAIN})")
ax.legend(loc="lower right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_cosine.png", dpi=150); plt.show()

In [ ]:
# [7] §4 scorecard for every arm + Fig 4 (Edit Index by rollout step).
ORDER = (["Unsteered"] + [f"Input Grad @L{L} (n=1, λ={LAM_MAIN})" for L in range(N_RESID)]
         + [f"Input Grad @L4 (n=1, λ={lam})" for lam in [0.0, 0.01, 1.0]]
         + [f"Input Grad @L4 (n=all {ef}, λ={LAM_MAIN})", "Render write @1 (oracle)"])
hdr = ("| arm | Edit Index (−1…+1) | edit-frame RMSE | target RMSE | ghost RMSE | collateral RMSE "
       "| GT-traj RMSE | fidelity ratio |\n|---|---|---|---|---|---|---|---|\n")
rows = []
for name in ORDER:
    c = CARDS[name]
    rows.append(f"| {name} | {c['edit_index']:+.2f} | {c['edit_frame_rmse']:.3f} | {c['target_rmse']:.3f} "
                f"| {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | {c['gt_traj_rmse']:.3f} "
                f"| {fidelity_ratio(c, CARDS['Unsteered']):.2f} |")
display(Markdown("**§4 scorecard** (canonical set, `editability_metrics.py`; RMSE in intensity units vs the "
                 "clean edited-world render; step 0 = frame ef)\n\n" + hdr + "\n".join(rows)))

# Fig 4 — does the edit hold? Edit Index at every rollout step, one series per arm (subset for legibility).
FIG4 = ["Unsteered", f"Input Grad @L0 (n=1, λ={LAM_MAIN})", f"Input Grad @L4 (n=1, λ={LAM_MAIN})",
        f"Input Grad @L4 (n=all {ef}, λ={LAM_MAIN})", "Render write @1 (oracle)"]
colors = ["#999999", "#56B4E9", "#0072B2", "#D55E00", "#009E73"]
fig, ax = plt.subplots(figsize=(8, 4))
for name, col in zip(FIG4, colors):
    ax.plot(CARDS[name]["edit_index_by_step"], color=col, lw=1.8, marker="o", ms=3.5, label=name)
ax.axhline(0, color="#555555", lw=0.8, ls=":")
ax.set_xlabel("rollout step (0 = edit frame ef)"); ax.set_ylabel("Edit Index (−1…+1)")
ax.set_ylim(-1.05, 1.05)
ax.set_title(f"Fig 4 — Edit Index by rollout step ({MODEL_LABEL})")
ax.legend(fontsize=8, loc="upper right"); style_ax(ax)
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_index_by_step.png", dpi=150); plt.show()

In [ ]:
# [8] Fig 5 — observation-space waterfall (canonical fixed spec; one helper, every waterfall through it).
N_CTX = 6
ctx_obs = edits.obs[:N, ef - N_CTX:ef, :].astype(np.float32)   # actual NOISY teacher-forced context
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
pre_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname):
    """col_bodies[c]: (N, L, R) rows BELOW the N_CTX context frames; every column its OWN free-run
    from step 0 (= frame ef). No shared teacher-forced ef row (banned; see CLAUDE.md)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0 * ncol, 3.4 * len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], axis=0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1,
                      interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX - 0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(pre_cx[smp]): ax.axvline(pre_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX + 7, N_CTX + 14])
                ax.set_yticklabels([ef - N_CTX, ef, ef + 7, ef + 14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0], [0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0], [0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0], [0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.965))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(fname, dpi=150, facecolor=DARK, bbox_inches="tight"); plt.show()

WF = ["Unsteered", f"Input Grad @L0 (n=1, λ={LAM_MAIN})", f"Input Grad @L4 (n=1, λ={LAM_MAIN})",
      f"Input Grad @L4 (n=all {ef}, λ={LAM_MAIN})", "Render write @1 (oracle)"]
titles = ["GT (sim clean obs)"] + [f"{n}\nEdit Index {CARDS[n]['edit_index']:+.2f}" for n in WF]
bodies = [gt_roll] + [ROLLS[n] for n in WF]
waterfall_grid(titles, bodies, SAMPLES,
               f"Fig 5 — free-run waterfalls: input-gradient steering arms vs references ({MODEL_LABEL})",
               f"{OUT}/fig5_waterfall.png")

## Current results (updated 2026-08-11)

- **The probe is fully driven**: residual 3.4–3.5 → 0.06–0.19 sim-units at every residual point (cell [4]) — the
  optimization always succeeds *as an optimization*.
- **But the perturbation is adversarial-dominated**: cos(δ*, Δ_true) ≈ +0.21…+0.27 (74–78°) — above the
  shuffled-pair chance (−0.008) but far from aligned; the raw first gradient is ≈ 0 cosine (88–90°) everywhere
  except the encoder port (+0.13). Fig 2 shows broadband spiky fuzz, not a moved bump; the ghost is not removed.
- **The generation barely notices**: Edit Index −0.69 (unsteered) → −0.50…−0.67 across all layers/λ/n; the ghost
  RMSE improves slightly (0.57 → 0.49) with a matching collateral degradation (0.127 → 0.145). λ and n=all are
  second-order. Fidelity stays ≈ 1.0 — the write is *ignored*, not destructive.
- **Same surface, oracle content: +0.27** (Render write @1). The gap −0.50 vs +0.27 is purely the *content* the
  gradient found vs the renderer's — the write surface itself works (inertia-limited, consistent with First Obs
  TF ≈ −0.08 on the GRU with one frame of evidence).
- Probe R² note: the standard MLP probe under-reads at N_PROBE=800 (encoder port MLP R² −0.34); linear probes
  (0.58–0.79) are the steering objects and are unaffected.

## Summary (interpretation — clearly marked as such)

**Readable ≠ input-controllable.** Even at the *input observation* — the one fully on-manifold-parameterizable
surface the model has — the frozen linear probe's gradient overwhelmingly points off the data manifold: it flips
the readout completely while moving the observation only ≈ 0.25-cosine toward what the edited world actually
looks like, and the dynamics respond by largely ignoring it. The oracle arm shows the bottleneck is not the write
surface but the *direction-finding*: content matching the data manifold (a render) moves belief substantially.
This extends the thread's readable≠controllable result from `h`-space to input space, and sharpens the motivation
for the DiT: a mechanism that *projects perturbations back to the manifold* (denoising steps after probe-guidance)
is exactly what this editor is missing.